# Quick-Run Smoke Test (Colab, Hybrid I/O)

This notebook is for end-to-end pipeline validation before long reruns.

It does **not** reproduce paper-quality results.
Instead, it verifies that all major steps run correctly:
- symlinked processed dataset root
- benchmark build
- YOLO export
- two-stage smoke train/eval with saved predictions
- YOLO smoke train/eval with saved predictions
- in-domain qualitative figure builder
- VNWoodKnot smoke train/eval with saved predictions
- VNWoodKnot qualitative figure builder


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi


In [ ]:
from pathlib import Path
import os

REPO_URL = 'https://github.com/ntkhanh98/wood-defect-q2.git'
REPO_DIR = Path('/content/wood-defect-q2')
SOURCE_PATH = Path('/content/drive/MyDrive/2.Work/1.PTIT/1.Cá nhân/2.Research/2026/wood-defected-q2/data/processed_for_server')
DEST_PATH = Path('/content/processed_for_server')

MAIN_ROOT = DEST_PATH / 'main_dataset'
VN_ROOT = DEST_PATH / 'vnwoodknot'
SMOKE_ROOT = Path('/content/drive/MyDrive/wood_q2_quick_run')

SMOKE_TARGET_SOURCE_IMAGES = 200
SMOKE_YOLO_EPOCHS = 1
SMOKE_T0_T1_EPOCHS = 1
SMOKE_TWO_STAGE_EPOCHS = 1
SMOKE_MAX_TRAIN_SAMPLES = 128
SMOKE_MAX_VAL_SAMPLES = 32
IMAGE_SIZE = 1024

os.environ['WOOD_MAIN_PROCESSED_ROOT'] = str(MAIN_ROOT)
os.environ['WOOD_VN_PROCESSED_ROOT'] = str(VN_ROOT)
os.environ['PYTHONUNBUFFERED'] = '1'

print('SOURCE_PATH =', SOURCE_PATH)
print('DEST_PATH   =', DEST_PATH)
print('MAIN_ROOT   =', MAIN_ROOT)
print('VN_ROOT     =', VN_ROOT)
print('SMOKE_ROOT  =', SMOKE_ROOT)


In [ ]:
%%bash
set -euo pipefail

cd /content
rm -rf wood-defect-q2
git clone "$REPO_URL" wood-defect-q2
cd wood-defect-q2

python3 -m pip install -q ultralytics==8.3.0 timm pycocotools pandas pillow pyyaml


In [ ]:
%%bash
set -euo pipefail

rm -rf /content/processed_for_server || true
ln -sfn "$SOURCE_PATH" "$DEST_PATH"
echo "Linked $SOURCE_PATH -> $DEST_PATH"


In [ ]:
SMOKE_ROOT.mkdir(parents=True, exist_ok=True)
(SMOKE_ROOT / 'yolo').mkdir(exist_ok=True)
(SMOKE_ROOT / 'tables').mkdir(exist_ok=True)
(SMOKE_ROOT / 'figures').mkdir(exist_ok=True)


## Build a small curated benchmark and local YOLO exports

In [ ]:
%%bash
set -euo pipefail

cd /content/wood-defect-q2

python3 scripts/build_screened_benchmark.py \
  --input-manifest /content/processed_for_server/main_dataset/manifest.jsonl \
  --output-root-dir /content/processed_for_server/main_dataset/benchmarks/vsb7_200_rare_first_smoke \
  --dataset-name large_scale_wood_surface_defects_vsb7_200_rare_first_smoke \
  --target-source-images 200 \
  --seed 42 \
  --classes live_knot dead_knot resin knot_with_crack crack marrow knot_missing \
  --selection-mode rare_first

rm -rf /content/local_data/main_dataset/benchmarks/vsb7_200_rare_first_smoke_yolo || true
rm -rf /content/local_data/vnwoodknot/benchmarks/vnwoodknot_live_dead_2class_yolo_smoke || true

python3 scripts/build_yolo_dataset.py \
  --input-manifest /content/processed_for_server/main_dataset/benchmarks/vsb7_200_rare_first_smoke/manifest.jsonl \
  --image-root-dir /content/processed_for_server/main_dataset \
  --output-root-dir /content/local_data/main_dataset/benchmarks/vsb7_200_rare_first_smoke_yolo \
  --dataset-name large_scale_wood_surface_defects_vsb7_200_rare_first_smoke_yolo \
  --classes live_knot dead_knot resin knot_with_crack crack marrow knot_missing \
  --copy-images

python3 scripts/build_yolo_dataset.py \
  --input-manifest /content/processed_for_server/vnwoodknot/manifest.jsonl \
  --image-root-dir /content/processed_for_server/vnwoodknot \
  --output-root-dir /content/local_data/vnwoodknot/benchmarks/vnwoodknot_live_dead_2class_yolo_smoke \
  --dataset-name vnwoodknot_live_dead_2class_yolo_smoke \
  --classes live_knot dead_knot \
  --copy-images


## Generate smoke configs and the Y1 model YAML

In [ ]:
import yaml
from textwrap import dedent

generated_dir = REPO_DIR / 'configs' / 'generated_colab_smoke'
models_dir = REPO_DIR / 'configs' / 'models'
generated_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

dataset_cfg = {
    'dataset_name': 'large_scale_wood_surface_defects_vsb7_200_rare_first_smoke',
    'root_dir': '${WOOD_MAIN_PROCESSED_ROOT}',
    'manifest_path': '${WOOD_MAIN_PROCESSED_ROOT}/benchmarks/vsb7_200_rare_first_smoke/manifest.jsonl',
    'classes': ['live_knot', 'dead_knot', 'resin', 'knot_with_crack', 'crack', 'marrow', 'knot_missing'],
}
dataset_cfg_path = generated_dir / 'dataset_main_vsb7_200_rare_first_smoke.yaml'
with dataset_cfg_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(dataset_cfg, f, sort_keys=False)

train_cfg = {
    'seed': 42,
    'device': 'cuda',
    'output_dir': str(SMOKE_ROOT),
    'experiment_name': 'baseline_mobilenet_hr_vsb7_200_rare_first_smoke',
    'dataset': {
        'train': 'configs/generated_colab_smoke/dataset_main_vsb7_200_rare_first_smoke.yaml',
        'val': 'configs/generated_colab_smoke/dataset_main_vsb7_200_rare_first_smoke.yaml',
        'train_split': 'train',
        'val_split': 'val',
    },
    'dataset_split': {'seed': 42, 'train_ratio': 0.8, 'val_ratio': 0.1},
    'train': {
        'epochs': 1,
        'batch_size': 2,
        'num_workers': 2,
        'image_size': int(IMAGE_SIZE),
        'learning_rate': 1e-4,
        'weight_decay': 1e-4,
        'best_metric': 'mAP50_95',
    },
    'model': {
        'name': 'baseline_detector',
        'num_classes': 7,
        'backbone': 'mobilenet_hr',
        'image_size': int(IMAGE_SIZE),
        'score_threshold': 0.05,
        'nms_threshold': 0.5,
        'max_detections': 100,
    },
}
eval_cfg = {
    'seed': 42,
    'device': 'cuda',
    'output_dir': str(SMOKE_ROOT),
    'experiment_name': 'baseline_mobilenet_hr_vsb7_200_rare_first_smoke_eval',
    'checkpoint_path': str(SMOKE_ROOT / 'checkpoints' / 'baseline_mobilenet_hr_vsb7_200_rare_first_smoke' / 'best.pt'),
    'dataset': {
        'eval': 'configs/generated_colab_smoke/dataset_main_vsb7_200_rare_first_smoke.yaml',
        'split': 'test',
    },
    'dataset_split': {'seed': 42, 'train_ratio': 0.8, 'val_ratio': 0.1},
    'model': {
        'name': 'baseline_detector',
        'num_classes': 7,
        'backbone': 'mobilenet_hr',
        'image_size': int(IMAGE_SIZE),
        'score_threshold': 0.05,
        'nms_threshold': 0.5,
        'max_detections': 100,
    },
    'evaluation': {
        'batch_size': 1,
        'num_workers': 2,
        'score_threshold': 0.05,
        'save_predictions': True,
        'save_visualizations': False,
        'compute_per_class_ap': True,
        'compute_cross_dataset': False,
    },
}

train_cfg_path = generated_dir / 'train_baseline_vsb7_200_rare_first_smoke.yaml'
eval_cfg_path = generated_dir / 'eval_baseline_vsb7_200_rare_first_smoke.yaml'
with train_cfg_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(train_cfg, f, sort_keys=False)
with eval_cfg_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(eval_cfg, f, sort_keys=False)

y1_model_yaml = dedent('''\
nc: 7
depth_multiple: 0.33
width_multiple: 0.50
backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]
head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 2], 1, Concat, [1]]
  - [-1, 3, C2f, [128]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 15], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 3, C2f, [1024]]
  - [[18, 21, 24, 27], 1, Detect, [nc]]
''')
y1_model_path = models_dir / 'yolov8s-p2-7class.yaml'
y1_model_path.write_text(y1_model_yaml, encoding='utf-8')

print(train_cfg_path)
print(eval_cfg_path)
print(dataset_cfg_path)
print(y1_model_path)


## Smoke test the in-domain figure pipeline

In [ ]:
%%bash
set -euo pipefail

cd /content/wood-defect-q2

python3 scripts/train.py \
  --config configs/generated_colab_smoke/train_baseline_vsb7_200_rare_first_smoke.yaml \
  --experiment-name baseline_mobilenet_hr_vsb7_200_rare_first_smoke \
  --epochs 1 \
  --max-train-samples 128 \
  --max-val-samples 32 \
  --device cuda

python3 scripts/evaluate.py \
  --config configs/generated_colab_smoke/eval_baseline_vsb7_200_rare_first_smoke.yaml \
  --checkpoint /content/drive/MyDrive/wood_q2_quick_run/checkpoints/baseline_mobilenet_hr_vsb7_200_rare_first_smoke/best.pt \
  --experiment-name baseline_mobilenet_hr_vsb7_200_rare_first_smoke_eval \
  --max-samples 32 \
  --device cuda

python3 scripts/train_yolov8.py \
  --data /content/local_data/main_dataset/benchmarks/vsb7_200_rare_first_smoke_yolo/dataset.yaml \
  --model yolov8s \
  --experiment-name y0_yolov8s_vsb7_200_rare_first_smoke \
  --epochs 1 \
  --imgsz 1024 \
  --batch 16 \
  --device 0 \
  --workers 2 \
  --seed 42 \
  --patience 5 \
  --project-dir /content/drive/MyDrive/wood_q2_quick_run/yolo

python3 scripts/evaluate_yolov8.py \
  --dataset-config configs/generated_colab_smoke/dataset_main_vsb7_200_rare_first_smoke.yaml \
  --checkpoint /content/drive/MyDrive/wood_q2_quick_run/yolo/y0_yolov8s_vsb7_200_rare_first_smoke/weights/best.pt \
  --experiment-name y0_yolov8s_vsb7_200_rare_first_smoke_eval \
  --split test \
  --batch 8 \
  --imgsz 1024 \
  --device 0 \
  --output-dir /content/drive/MyDrive/wood_q2_quick_run \
  --save-predictions

python3 scripts/train_yolov8.py \
  --data /content/local_data/main_dataset/benchmarks/vsb7_200_rare_first_smoke_yolo/dataset.yaml \
  --model /content/wood-defect-q2/configs/models/yolov8s-p2-7class.yaml \
  --experiment-name y1_yolov8s_p2_vsb7_200_rare_first_smoke \
  --epochs 1 \
  --imgsz 1024 \
  --batch 8 \
  --device 0 \
  --workers 2 \
  --seed 42 \
  --patience 5 \
  --project-dir /content/drive/MyDrive/wood_q2_quick_run/yolo

python3 scripts/evaluate_yolov8.py \
  --dataset-config configs/generated_colab_smoke/dataset_main_vsb7_200_rare_first_smoke.yaml \
  --checkpoint /content/drive/MyDrive/wood_q2_quick_run/yolo/y1_yolov8s_p2_vsb7_200_rare_first_smoke/weights/best.pt \
  --experiment-name y1_yolov8s_p2_vsb7_200_rare_first_smoke_eval \
  --split test \
  --batch 8 \
  --imgsz 1024 \
  --device 0 \
  --output-dir /content/drive/MyDrive/wood_q2_quick_run \
  --save-predictions

python3 scripts/build_in_domain_qualitative_figure.py \
  --manifest /content/processed_for_server/main_dataset/benchmarks/vsb7_200_rare_first_smoke/manifest.jsonl \
  --image-root-dir /content/processed_for_server/main_dataset \
  --split test \
  --rows 2 \
  --baseline-run-name baseline_mobilenet_hr_vsb7_200_rare_first_smoke \
  --baseline-header 'Faster R-CNN' \
  --baseline-predictions /content/drive/MyDrive/wood_q2_quick_run/tables/baseline_mobilenet_hr_vsb7_200_rare_first_smoke_eval_test_predictions.jsonl \
  --yolo-run-name y0_yolov8s_vsb7_200_rare_first_smoke \
  --yolo-header YOLOv8s \
  --yolo-predictions /content/drive/MyDrive/wood_q2_quick_run/tables/y0_yolov8s_vsb7_200_rare_first_smoke_eval_test_predictions.jsonl \
  --variant-run-name y1_yolov8s_p2_vsb7_200_rare_first_smoke \
  --variant-header 'YOLO P2' \
  --variant-predictions /content/drive/MyDrive/wood_q2_quick_run/tables/y1_yolov8s_p2_vsb7_200_rare_first_smoke_eval_test_predictions.jsonl \
  --output-dir /content/drive/MyDrive/wood_q2_quick_run/figures/in_domain_smoke


## Smoke test the VNWoodKnot figure pipeline

In [ ]:
%%bash
set -euo pipefail

cd /content/wood-defect-q2

python3 scripts/train_yolov8.py \
  --data /content/local_data/vnwoodknot/benchmarks/vnwoodknot_live_dead_2class_yolo_smoke/dataset.yaml \
  --model yolov8s \
  --experiment-name t0_vnwoodknot_smoke \
  --epochs 1 \
  --imgsz 1024 \
  --batch 16 \
  --device 0 \
  --workers 2 \
  --seed 42 \
  --patience 5 \
  --project-dir /content/drive/MyDrive/wood_q2_quick_run/yolo

python3 scripts/train_yolov8.py \
  --data /content/local_data/vnwoodknot/benchmarks/vnwoodknot_live_dead_2class_yolo_smoke/dataset.yaml \
  --weights /content/drive/MyDrive/wood_q2_quick_run/yolo/y0_yolov8s_vsb7_200_rare_first_smoke/weights/best.pt \
  --experiment-name t1_vnwoodknot_smoke \
  --epochs 1 \
  --imgsz 1024 \
  --batch 16 \
  --device 0 \
  --workers 2 \
  --seed 42 \
  --patience 5 \
  --project-dir /content/drive/MyDrive/wood_q2_quick_run/yolo

python3 scripts/evaluate_yolov8.py \
  --dataset-config configs/dataset_transfer.yaml \
  --checkpoint /content/drive/MyDrive/wood_q2_quick_run/yolo/t0_vnwoodknot_smoke/weights/best.pt \
  --experiment-name t0_vnwoodknot_smoke_eval \
  --split test \
  --batch 8 \
  --imgsz 1024 \
  --device 0 \
  --output-dir /content/drive/MyDrive/wood_q2_quick_run \
  --save-predictions

python3 scripts/evaluate_yolov8.py \
  --dataset-config configs/dataset_transfer.yaml \
  --checkpoint /content/drive/MyDrive/wood_q2_quick_run/yolo/t1_vnwoodknot_smoke/weights/best.pt \
  --experiment-name t1_vnwoodknot_smoke_eval \
  --split test \
  --batch 8 \
  --imgsz 1024 \
  --device 0 \
  --output-dir /content/drive/MyDrive/wood_q2_quick_run \
  --save-predictions

python3 scripts/build_vnwoodknot_qualitative_from_predictions.py \
  --manifest /content/processed_for_server/vnwoodknot/manifest.jsonl \
  --split test \
  --rows 2 \
  --t0-run-name t0_vnwoodknot_smoke \
  --t0-header T0 \
  --t0-predictions /content/drive/MyDrive/wood_q2_quick_run/tables/t0_vnwoodknot_smoke_eval_test_predictions.jsonl \
  --t1-run-name t1_vnwoodknot_smoke \
  --t1-header T1 \
  --t1-predictions /content/drive/MyDrive/wood_q2_quick_run/tables/t1_vnwoodknot_smoke_eval_test_predictions.jsonl \
  --output-dir /content/drive/MyDrive/wood_q2_quick_run/figures/vnwoodknot_smoke


In [ ]:
%%bash
set -euo pipefail

echo 'In-domain smoke outputs:'
find /content/drive/MyDrive/wood_q2_quick_run/figures/in_domain_smoke -maxdepth 1 -type f | sort || true
echo
echo 'VNWoodKnot smoke outputs:'
find /content/drive/MyDrive/wood_q2_quick_run/figures/vnwoodknot_smoke -maxdepth 1 -type f | sort || true
